## fine tune custom dataset with Unsloth Framework.
#####fine tune model for Text generation (https://unsloth.ai/)

In [ ]:
!pip install "unsloth[colab] @git+https://github.com/unslothai/unsloth.git"

In [ ]:
# huggingface transformer

!pip install "git+https://github.com/huggingface/transformers.git"

In [ ]:
# Train transformer language models with reinforcement learning. # https://github.com/huggingface/trl
!pip install trl

### Setup Unsloth FastLanguageModel

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

In [ ]:
# Load huggingface datasets
from datasets import load_dataset
max_seq_length = 2048
# loading huggingface.co/datasets/imdb
dataset = load_dataset("imdb", split="train")


In [ ]:
dataset

### Model and Tokenizer
*   Mistral
* bitsandbyters (bnb)
*   4-bit quantized model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True
)

### Get PEFT Model
*   Model pacthing
*   Add Fast Lora weights

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length
)

### Supervised Finetune trainer

In [ ]:
from trl import SFTTrainer, SFTConfig #Import SFTConfig
from transformers import TrainingArguments
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
       # dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "unsloth-test",
        optim = "adamw_8bit",
        seed = 3407,
        max_steps=20
    )
)

trainer.train()

### Test the model

In [ ]:
# create a tokenizer
inputs = tokenizer(
    [
        "I really like the movie because it talks about SCI-FI, Anime, humanity and human emotions"
    ],
    return_tensors ="pt"
).to("cuda")

print(f" input ids ==> {inputs}")

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
outputs

In [ ]:
# decode the tokenizer
tokenizer.batch_decode(outputs)

In [ ]:
# Save the model
model.save_pretrained("unsloth_finetune_model")